In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(".."))

import models.utils as utils 
import models.agents.uni_random as uni_random
import models.agents.heuristics as heuristics
import models.agents.mcts as mcts
import analysis.analysis_utils as analysis_utils
import matplotlib.pylab as plt 
import seaborn as sns
import pandas as pd
import random
import pickle
import importlib
import constants
import numpy as np
import json
from scipy.stats import pearsonr
from models.agents.uni_random import uniform_dist

%matplotlib inline


save_dir = "watch_figs/"
main_final_save_dir = "final_processed_res/watch/"


In [ ]:
""" 
Init constants
"""
importlib.reload(constants)
model2name = constants.MODEL2NAME
model2pth = constants.MODEL2PTH["think"]

ordered_models = ["ours", "random",]

"""
Load in games, and filter to include just the games from the live gameplay study
"""
games, idx2game, game2idx, game_stimuli = analysis_utils.process_game_stimuli(constants.THINK_STIMULI_PTH)
livegameplay_games = pd.read_csv(constants.PLAY_STIMULI_PTH)
livegameplay_games["stimuli_id"] = [analysis_utils.get_stimuli_id(row) for idx, row in livegameplay_games.iterrows()]
play_game_objs = []
played_game_ids = set()
for game_id in livegameplay_games["stimuli_id"]:
    game = game_stimuli[game_id]
    played_game_ids.add(game["index"])
    game["stimuli_id"] = game_id
    play_game_objs.append(game)

"""
Load in human data
"""
with open(f"../human-data/play-exp/human-v-human/final-play/final_agg.json", "r") as f: 
    human_gameplay_data = json.load(f)

think_human_df = pd.read_csv(constants.THINK_HUMAN_DATA)
# pulling out games according to Ced's game categories
all_game_types = {}
for i, entry in think_human_df.iterrows(): 
    game_type = entry.game_types
    game_id = entry.game_id
    if game_type not in all_game_types: all_game_types[game_type] = {game_id}
    else: all_game_types[game_type].add(game_id)

"""
Load in model preds
"""

model2res = {model: [] for model in ordered_models}
for model, model_pth in model2pth.items(): 
    res = analysis_utils.process_model_runs(model, model_pth, idx2game)
    if res is not None: model2res[model] = res


'''
Load in pre-run model fits to people
'''
model_fit_data = {}

with open("../model-data/play-exp/final_data/heuristics.txt", "r") as f: 
    model_fit_data['ours'] = eval(f.readlines()[0])

expert_pickle_file_path = "../model-data/play-exp/2025-07-02_heuristic_search_eg.pickle"
with open(expert_pickle_file_path, 'rb') as file:
    loaded_data = pickle.load(file)
model_fit_data['expert'] = loaded_data

ours_depth_3pickle_file_path = '../model-data/play-exp/final_data/depth3-new-may24.pickle' 
with open(ours_depth_3pickle_file_path, 'rb') as file:
    loaded_data = pickle.load(file)

model_fit_data['ours-depth3'] = loaded_data

''' 
Human watch data
'''

with open("../human-data/watch-exp/final_res.json", "r") as f:
    watch_data = json.load(f)

print("num uids: ", len(watch_data))

# filtered from post-judgements
rem_ids = ['MUJ2ELF5D6',
 'XAOHNXHKQL',
 'FRF6G0BXYN',
 'J090IDCESB',
 'ZMSROEENK1',
 'ZRSSZKF1VU',
 'B8RXDIXOCL',
 'M9SW86IBCL',
 '2WKOW6RBSH',
 'O5CGUJIT5A']



In [ ]:
''' 
Extract key info from the original human game 

{game: {arena: { 
}
}
'''

game_stages = ['early', 'intermediate', 'late']
order2stage = {i: v for i, v in enumerate(game_stages)}

move_idx_offset = -1

played_game_data = {}

for uid, uid_data in watch_data.items(): 
    if uid in rem_ids: continue
    for game, game_res in uid_data.items(): 
        batch_metadata = game_res['batch_metadata']
        arena = batch_metadata['arena'] 
        
        played_moves = batch_metadata['played_pos']
        
        if game not in played_game_data: played_game_data[game] = {}
        if arena not in played_game_data[game]: played_game_data[game][arena] = {}
        
        # get moves 'so far' for each board 
        boards = batch_metadata['boards']
        processed_boards = analysis_utils.process_game_states(boards)
        pred_move_idxs = batch_metadata['pred_move_idxs']
        
        # get game outcome 
        play_data = human_gameplay_data[game]
        player2order = None
        # make sure to pull out the "right" arena
        for arena2, order2player, outcome, all_move_times in zip(play_data['arena'],  
                                                 play_data["order2player"], 
                                                 play_data["outcome"],
                                                 play_data['move_times']): 
            if arena != arena2: continue
            player2order = {v: k for k,v in order2player.items()}

            if outcome == "Draw": 
                obs_outcome = 0 
            else: 
                obs_outcome = int(player2order[outcome])
                
            played_move_times = [all_move_times[idx] for idx in pred_move_idxs]
                
            break
        
        boards_at_stage = {}
        
        for order, move_idx in enumerate(pred_move_idxs): 
            stage = game_stages[order]
            boards_at_stage[stage] = processed_boards[move_idx + move_idx_offset]
            
        played_game_data[game][arena] = {
            'boards_at_stage': boards_at_stage,
            'played_pos': played_moves,
            'obs_outcome': obs_outcome,
            'move_idxs': pred_move_idxs, 
            'player2order': player2order, 
            'played_move_times': played_move_times,
            'player_at_pos': batch_metadata['player_at_idx']
            
        }
        
all_games = sorted(played_game_data.keys())
                

In [ ]:

'''
Process the human watch data
''' 
human_watch_data = {game: {} for game in all_games}
human_watch_payoffs = {game: [] for game in all_games}
human_watch_fun = {game: [] for game in all_games}

for user, user_data in watch_data.items(): 
    for game, game_data in user_data.items(): 
        arena = game_data['batch_metadata']['arena']
        
        if arena not in human_watch_data[game]: 
            human_watch_data[game][arena] = {game_stage: {
                'dist': [], 'rt': [], 'uid': []
                } for game_stage in game_stages}
            
        payoff = game_data['judgements'][-1]
        if payoff is not None: 
            human_watch_payoffs[game].append(payoff)
        else: 
            if 'response' in game_data['judgements'][0]: human_watch_fun[game].append(game_data['judgements'][0]['response'])
            
        for order, entry in enumerate(game_data['pred_moves']):
            raw_pred_moves, _, _, _, rt = entry
            game_stage = game_stages[order]
            
            # convert click count format of keys: 'row-col' --> (row, col)
            pred_move_dist = {}
            n_rows_game, n_cols_game, win_conds = game.split("*")
            n_rows_game = int(float(n_rows_game))
            n_cols_game = int(float(n_cols_game))
            
            for r in range(n_rows_game): 
                for c in range(n_cols_game): 
                    pred_move_dist[(r, c)] = 0
            norm_counts = int(np.sum(list(raw_pred_moves.values())))
            for k, v in raw_pred_moves.items(): 
                row, col = k.split('-')
                row = int(row)
                col = int(col)
                k = (row,col)
                pred_move_dist[k] = v / norm_counts

            human_watch_data[game][arena][game_stage]['dist'].append(pred_move_dist)
            human_watch_data[game][arena][game_stage]['rt'].append(rt)
            human_watch_data[game][arena][game_stage]['uid'].append(user)



In [ ]:
play_times = []
watch_times =[] 
watch_times_se = []
board_sizes = []
for game, entries in played_game_data.items():
    n_rows_game, n_cols_game, win_conds = game.split("*")
    n_rows_game = int(float(n_rows_game))
    n_cols_game = int(float(n_cols_game))
    n_cells = n_rows_game * n_cols_game
    for arena, arena_entries in entries.items():
        pt = arena_entries['played_move_times']
        for stage_idx, stage in enumerate(game_stages): 
            rts = human_watch_data[game][arena][game_stage]['rt']
            play_time = pt[stage_idx + move_idx_offset] # check indexing
            
            if np.mean(rts) > 70: 
                print(game, stage, play_time, np.mean(rts))
                
            if analysis_utils.compute_se(rts) > 20: 
                print(game, stage, play_time, np.mean(rts), analysis_utils.compute_se(rts), "\n",rts)
            play_times.append(play_time)
            watch_times.append(np.median(rts))
            watch_times_se.append(analysis_utils.compute_se(rts))
            board_sizes.append(n_cells)

print(pearsonr(play_times, watch_times))
print(pearsonr(board_sizes, play_times))
print(pearsonr(board_sizes, watch_times))
plt.figure()
plt.scatter(play_times, watch_times, alpha=0.5)
plt.errorbar(play_times, watch_times, yerr=watch_times_se, fmt='o', alpha=0.3)
plt.xlabel("Play Response Time", fontsize=18)
plt.ylabel("Watch Response Time", fontsize=18)

plt.figure()
plt.scatter(board_sizes, watch_times)
plt.errorbar(board_sizes, watch_times, yerr=watch_times_se, fmt='o')
plt.xlabel("Board Size", fontsize=18)
plt.ylabel("Watch Response Time", fontsize=18)


In [ ]:
all_agents = ['ours', 
              'ours-depth3',
          'expert', 
          'random']

sub_agents = ['ours', 
          'expert', 
          'random']

comp_play_agents = ['should',
                    'ours', 
                    'ours-depth3',
          'expert', 
          'random']

agent2viz = {'ours': 'Novice', 
             'should': 'Human Watcher',
          'expert': 'Expert', 
          'split-human': 'Split-Half\nHuman', 
          'ours-depth3': 'Novice\n(Depth-3)',
          'expert-depth1': 'Global Eval\n(Depth-1)',
          'random': 'Random'}


agent2color = {'expert': 'blue', 
               'split-human': 'green', 
          'ours': 'red', 
          'should': 'pink', 
          'ours-depth3': 'orange',
          'expert-depth1': 'purple',
          'random': 'grey'}

In [ ]:

''' 
Process the model data 
'''
agents = all_agents
model_watch_data = {agent: 
    {game: {} for game in played_game_data.keys()}
    for agent in agents
    }
model_watch_data_nosoftmax = {agent: 
    {game: {} for game in played_game_data.keys()}
    for agent in agents
    }

for agent in agents: 
    for game in played_game_data.keys(): 
        game_id = game2idx[game]
        game_obj = games[game_id - 1]
        
        for arena, arena_data in played_game_data[game].items(): 
            model_watch_data[agent][game][arena] = {game_stage: [] for game_stage in game_stages}
            model_watch_data_nosoftmax[agent][game][arena] = {game_stage: [] for game_stage in game_stages}
            boards = arena_data['boards_at_stage']
            
            move_idxs = arena_data['move_idxs']
            
            for order, game_stage in enumerate(game_stages): 
                
                move_idx = move_idxs[order] + move_idx_offset
                board = boards[game_stage]
                # just take the plays 
                board = [[piece for piece, move_turn in entries] for entries in board]
                board = analysis_utils.convert_board_repr(board)
                    
                if agent != "random": 
                    raw_m_outputs = model_fit_data[agent][arena][str(game_obj['index'])][str(move_idx)]['dist']
                    if type(raw_m_outputs) != list: raw_m_outputs =[raw_m_outputs]
                    
                    m_outputs = []
                    for m_output in raw_m_outputs: 
                        new_dict = {}
                        for k, v in m_output.items():
                            if v != -np.inf: new_dict[k] = v
                        m_outputs.append(new_dict)
                    
                    
                    m_dist = [utils.softmax_dist(m_output, T=1) for m_output in m_outputs]
                else: 
                    
                    # get all possible legal moves, and use this to ground comparisons
                    m_dist = [uniform_dist(board)]
                    m_outputs = m_dist # same for random
                    
                model_watch_data[agent][game][arena][game_stage] = m_dist
                model_watch_data_nosoftmax[agent][game][arena][game_stage] = m_outputs
            

In [ ]:

importlib.reload(analysis_utils)
metrics = [
    'TV',
           'JSD'
           ]

def get_board_corr(h_dist, m_dist,alpha=0.8): 
    num_moves = len(m_dist.keys())
    m = [] 
    h = []
    for k, v in m_dist.items():
        h_p = h_dist[k]
        h.append(h_p)
        m_p = alpha*v+(1-alpha)/num_moves
        m.append(m_p)
    return pearsonr(m, h)[0]


def get_board_lists(h_dist, m_dist,alpha=0.8): 
    num_moves = len(m_dist.keys())
    m = [] 
    h = []
    for k, v in m_dist.items():
        h_p = h_dist[k]
        h.append(h_p)
        m_p = alpha*v+(1-alpha)/num_moves
        m.append(m_p)
    return m, h


metric2func = {'KL': analysis_utils.get_kl,
                  'TV': analysis_utils.get_tv,
                  }
n_samples_per_move = 10

considered_games = sorted(list(human_watch_data))
all_participant_data = {stim: {} for stim in considered_games}
all_model_data = {agent: {stim: {} for stim in considered_games} for agent in all_agents}
all_per_participant_data = {}


# for metric in metrics: 

metric_data = {
    'game': [],
    'arena': [],
    'stage': [],
    'agent': [],
    'human_data_type': [],
    'alpha': []
}
agents = all_agents
metric_data.update({metric: [] for metric in metrics})

np.random.seed(7)
random.seed(7)

agg_human_dists_watch ={}

step = 0.05
alpha_vals= list(np.arange(0.5, 1.0, step))
print(alpha_vals)
agg_human= True

agent2corrs = {agent: {} for agent in agents}

for alpha in alpha_vals: 
    agent2lists ={agent: [] for agent in agents}
    for game, game_data in played_game_data.items():
        for arena, arena_data in game_data.items(): 
            # arena_data has data for all stages
            
            played_moves = arena_data['played_pos']
            
            for order, game_stage in enumerate(game_stages): 
                h_stage_entries = human_watch_data[game][arena][game_stage]
                h_dists = h_stage_entries['dist']
                
                # add each to the per participant dist
                for h_dist, uid in zip(h_dists, h_stage_entries['uid']): 
                    
                    all_per_participant_data.setdefault(uid, {})
                    print("adding uid: ", uid)
                    all_per_participant_data[uid].setdefault(game, {})
                    all_per_participant_data[uid][game].setdefault(arena, {})
                    all_per_participant_data[uid][game][arena][order] = h_dist
                    
                if agg_human: h_dists = [analysis_utils.average_dicts(h_dists)]
                    
                all_participant_data[game].setdefault(arena, {})
                all_participant_data[game][arena][order] = h_dists[0] # if agg, just one


                draw_samples = False

                for agent in agents:  
                    
                    if agent == 'ours' and alpha == 0.5: 
                        agg_human_dists_watch.setdefault(game, {})
                        agg_human_dists_watch[game].setdefault(game_stage, {})
                        agg_human_dists_watch[game][game_stage].setdefault(arena, {})
                        agg_human_dists_watch[game][game_stage][arena] = h_dists[0]
                    
                    
                    m_dists = model_watch_data[agent][game][arena][game_stage]
                    m_dists_nosoftmax = model_watch_data_nosoftmax[agent][game][arena][game_stage]
                    draw_samples = True
                    
                    all_model_data[agent][game].setdefault(arena, {})
                    all_model_data[agent][game][arena][order] = m_dists_nosoftmax # no alpha yet
                    
            
                    # compute metrics 
                    scores = []
                    for metric_idx, metric in enumerate(metrics):    
                        m_dist = m_dists[0]
                        h_dist = h_dists[0]
                        for k,vh in h_dist.items(): 
                            if k not in m_dist and vh != 0: print(k, vh)
                        if metric == 'KL':
                            scores = [np.mean([analysis_utils.get_kl(h_dist, m_dist, alpha=alpha) for m_dist in m_dists]) for h_dist in h_dists]
                        elif metric == 'TV':
                            scores = [np.mean([analysis_utils.get_tv(h_dist, m_dist, alpha=alpha) for m_dist in m_dists])  for h_dist in h_dists]
                        elif metric == 'corr':
                            scores = [np.mean([get_board_corr(h_dist, m_dist, alpha=alpha) for m_dist in m_dists])  for h_dist in h_dists]
                        elif metric == 'JSD': 
                            scores = [np.mean([analysis_utils.get_jsd(h_dist, m_dist, alpha=alpha) for m_dist in m_dists])  for h_dist in h_dists]
                        
                        if metric_idx == 0: 
                            metric_data['game'].extend([game for _ in range(len(h_dists))])
                            metric_data['arena'].extend([arena for _ in range(len(h_dists))])
                            metric_data['agent'].extend([agent for _ in range(len(h_dists))])
                            metric_data['stage'].extend([game_stage for _ in range(len(h_dists))])
                            metric_data['human_data_type'].extend(['should' for _ in range(len(h_dists))])
                            metric_data['alpha'].extend([alpha for _ in range(len(h_dists))])

                        for score in scores:
                            metric_data[metric].append(score)
for k, v in metric_data.items():
    print(k, len(v))

# also add split-half res
split_bootstrap = 100
for alpha in [0.99999]:
    for game, game_data in played_game_data.items():
        for arena, arena_data in game_data.items(): 
            
            n_h = len(human_watch_data[game][arena]['early']['dist'])
            h_idxs = np.arange(n_h)
            
            played_moves = arena_data['played_pos']
            
            split_stages = {game_stage: [] for game_stage in game_stages}
            
            for split_idx in range(split_bootstrap):  
                
                h1 = np.random.choice(h_idxs, size = int(n_h/2), replace=False)
                h2 = [h_idx for h_idx in h_idxs if h_idx not in h1]
                for order, game_stage in enumerate(game_stages): 
                    
                    h_dists1 = [dist for h_idx, dist in enumerate(human_watch_data[game][arena][game_stage]['dist']) if h_idx in h1]
                    h_dists2 = [dist for h_idx, dist in enumerate(human_watch_data[game][arena][game_stage]['dist']) if h_idx in h2]
                    if agg_human: 
                        h_dists1 = [analysis_utils.average_dicts(h_dists1)]
                        h_dists2 = [analysis_utils.average_dicts(h_dists2)]
                    
                    # compute metrics 
                    scores = []
                    for metric_idx, metric in enumerate(metrics): 
                        if metric == 'KL':
                            scores = [np.mean([analysis_utils.get_kl(h_dist1, h_dist2, alpha=alpha) for h_dist2 in h_dists2]) for h_dist1 in h_dists1]
                        elif metric == 'TV':
                            scores = [np.mean([analysis_utils.get_tv(h_dist1, h_dist2, alpha=alpha) for h_dist2 in h_dists2])  for h_dist1 in h_dists1]
                        elif metric == 'JSD': 
                            scores = [np.mean([analysis_utils.get_jsd(h_dist1, h_dist2, alpha=alpha) for h_dist2 in h_dists2])  for h_dist1 in h_dists1]
                            
                        split_stages[game_stage].append(scores[0]) 

                        if metric_idx == 0: 
                            metric_data['game'].extend([game for _ in range(len(h_dists))])
                            metric_data['arena'].extend([arena for _ in range(len(h_dists))])
                            metric_data['agent'].extend(['split-human' for _ in range(len(h_dists))])
                            metric_data['stage'].extend([game_stage for _ in range(len(h_dists))])
                            metric_data['human_data_type'].extend(['should' for _ in range(len(h_dists))])
                            metric_data['alpha'].extend([alpha for _ in range(len(h_dists))])

                        for score in scores:
                            metric_data[metric].append(score)

metric_df = pd.DataFrame.from_dict(metric_data)
metric_df.to_csv(f"{main_final_save_dir}metrics_agg.csv")

In [ ]:
'''
Admixture per game
'''
import admixture_utils as pf_utils
importlib.reload(pf_utils)
import random
np.random.seed(7)
random.seed(7)

arena_subset_prop = -1
game_subset_prop = -1
n_bootstrap_samples = 1

temp_step = 0.5
temp_vals= np.arange(0.5, 3.0 + temp_step, temp_step)
viz_agents = ['ours', 'expert', 'random']
admixture_agents = [agent for agent in viz_agents if agent != 'random'] # random automatically included

game_bootstrap_res_temp = {}

for i, game in enumerate(all_participant_data): 
    print("game: ", game)
    game_bootstrap_res_temp.setdefault(game, {temp: [] for temp in temp_vals})

    for temp in temp_vals: 
        bootres = []
        for _ in range(n_bootstrap_samples):
            res = pf_utils.optimize_mixture_weights_watch(all_model_data, all_participant_data,
                                                    model_names=admixture_agents,
                    arena_subset_prop=arena_subset_prop, game_id=game,
                     temperature=temp)
            bootres.append(res)
        game_bootstrap_res_temp[game][temp] = bootres
        
with open(f"{main_final_save_dir}per_game_admixture_watch.json", "w") as f: 
    json.dump(game_bootstrap_res_temp, f)


In [ ]:

importlib.reload(analysis_utils)
model2name = {'ours': 'Intuitive Gamer', 'random': 'Random', 'expert': 'Expert'}
per_game_df, fig, _ = analysis_utils.plot_individ_bars(game_bootstrap_res_temp, [agent for agent in viz_agents if agent != 'ours-depth3'],
                                                    filetag="bootstrap_game_weights_watch.png", label_x=[True, 'Games'], 
                                                    main_save="./",model2name=model2name, agent2color=agent2color)


In [ ]:

'''
Admixture per game
'''
import admixture_utils as pf_utils
importlib.reload(pf_utils)
import random
np.random.seed(7)
random.seed(7)

arena_subset_prop = -1
game_subset_prop = -1
n_bootstrap_samples = 1

temp_step = 0.5
temp_vals= np.arange(0.5, 3.0 + temp_step, temp_step)
viz_agents = ['ours', 'expert', 'random']
admixture_agents = [agent for agent in viz_agents if agent != 'random'] # random automatically included

participant_bootstrap_res_temp = {}

for i, uid in enumerate(all_per_participant_data): 
    participant_games_data = all_per_participant_data[uid] # all games for that participant
    if i % 50 == 0: print("On participant: ", i)
    participant_bootstrap_res_temp.setdefault(uid, {temp: [] for temp in temp_vals})

    for temp in temp_vals: 
        bootres = []
        for _ in range(n_bootstrap_samples):
            res = pf_utils.optimize_mixture_weights_watch(all_model_data, participant_games_data,
                                                    model_names=admixture_agents,
                    arena_subset_prop=arena_subset_prop, game_subset_prop=game_subset_prop, 
                     temperature=temp)
            bootres.append(res)
        participant_bootstrap_res_temp[uid][temp] = bootres

with open(f"{main_final_save_dir}per_uid_admixture_watch.json", "w") as f: 
    json.dump(participant_bootstrap_res_temp, f)